# 02 · Treinamento GRPO replicado (v4)

**Ambiente: GPU (T4 ou superior).** Executar tudo, com o mesmo `STUDY_ID`.

Treina none, cot_monitor e summary_monitor × níveis low/high × 20 sementes = 120 treinos de
120 passos. O monitor de recompensa (Qwen2.5-0.5B-Instruct) é distinto do monitor independente.
Reexecute até concluir; checkpoints são retomados e treinos concluídos são preservados.


In [ ]:
# Configuração operacional. Sementes, tamanhos, modelos e critérios estão fixos no protocolo v4.
STUDY_ID = "estudo-prospectivo-2026-v4"  # @param {type:"string"}
STORAGE_MODE = "auto"  # @param ["auto", "local"]
TECHNICAL_TEST = False  # @param {type:"boolean"}
MAX_JOBS_PER_SESSION = 1  # @param {type:"integer"}
CODE_COMMIT = "1f1d38d60b4d3fac85c702d86306a0c9e867dd77"
CODE_HASHES = {
 "__init__.py": "903b3bdb88f326693084e14633a5ebce1f862506124b58f67dc751c2a2c96f76",
 "avaliacao.py": "ce6219adc47aa9457db6d1923edd7d25128115c8e3dda7ca31835026dbfb54e0",
 "contrato.py": "ba3090c80e336ac7a34f1807348e50390e0bf5f1f6729b93b4c3395d6c8c0589",
 "estatistica.py": "7fd7615c0ead1e01c0d7e926e41c56a0b203b4ba493f3153362a9779c945bdfd",
 "modelos.py": "ee02866b8eac6db6928233805f50c985352d5c57253a3bfcf01d7c56fe910811",
 "monitor.py": "a6c94e7cc616523b9f83f94140c22f95110a50cb61ff5483e4bad9064a480a3d",
 "runtime.py": "2a66ab1a970dccbbd24a309b69f71f8095b1c554707fa73968bcd3070f04448a",
 "tarefas.py": "8b1a58ecc9df6b959741409f71811b7e90449bc41eedf7d860528f74732ba7fd",
 "treino.py": "39278193c27fefdeca9583a5e2fbf1225d81c3a75421dcfe20298abdb96d8ef3"
}
SUPPORT_HASHES = {
 "armazenamento.py": "b26088c7e2339b0c9968cad6f943db05a4effe27aa9eb5262bf84830e361f1f0",
 "sessao.py": "d35a4b3a51ed9216a8163e09368cf96a9ed0b8570f407446471f284e8808dbdb"
}

import hashlib, importlib.util, os, re, subprocess, sys, time
from pathlib import Path
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", STUDY_ID):
    raise ValueError("STUDY_ID aceita somente letras, números, _ e -.")
if TECHNICAL_TEST and not STUDY_ID.endswith("-teste-tecnico"):
    STUDY_ID += "-teste-tecnico"  # pasta separada: nunca se mistura ao estudo
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    drive, ON_COLAB = None, False
LOCAL = Path(os.environ.get("COT_LOCAL_ROOT", "/content/cot-v4" if ON_COLAB else Path.cwd() / "cot-v4"))
LOCAL.mkdir(parents=True, exist_ok=True)

def checkout(ref):
    """Obtém exatamente o commit fixado; não usa clones antigos nem a branch atual."""
    override = os.environ.get("COT_REPO_DIR")
    if override:
        return Path(override).resolve()
    repo = LOCAL / "source" / ref
    git = lambda *a, **k: subprocess.run(["git", "-C", str(repo), *a], capture_output=True, text=True, **k)
    if not (repo / ".git").is_dir():
        repo.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "--quiet", str(repo)], check=True)
        git("remote", "add", "origin", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git")
    if git("cat-file", "-e", ref + "^{commit}").returncode:
        for attempt in range(1, 4):
            fetched = git("fetch", "--quiet", "--depth", "1", "origin", ref)
            if not fetched.returncode:
                break
            print(f"Falha de rede ao baixar o código (tentativa {attempt}/3): {fetched.stderr.strip()[-300:]}")
            time.sleep(5 * attempt)
        else:
            raise RuntimeError("Não foi possível baixar o commit " + ref + ". Verifique a conexão e execute de novo.")
    if git("checkout", "--quiet", "--detach", ref).returncode:
        raise RuntimeError("Checkout do commit fixado falhou: " + ref)
    if git("rev-parse", "HEAD").stdout.strip() != ref:
        raise RuntimeError("O código aberto não corresponde ao commit do notebook.")
    return repo

REPO = checkout(CODE_COMMIT)
PACKAGE = REPO / "experimento_cientifico_v4"
def load(name):
    path = PACKAGE / "colab" / name
    if hashlib.sha256(path.read_bytes()).hexdigest() != SUPPORT_HASHES[name]:
        raise RuntimeError("Suporte divergente do notebook: " + name + ". Reabra pelo link do GitHub.")
    # Nome único por commit: módulos de versões antigas na mesma sessão não são reutilizados.
    spec = importlib.util.spec_from_file_location(f"cotv4_{name[:-3]}_{CODE_COMMIT[:12]}", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
armazenamento, sessao = load("armazenamento.py"), load("sessao.py")
STATE = armazenamento.prepare_storage(LOCAL, STUDY_ID, on_colab=ON_COLAB, mode=STORAGE_MODE,
    storage_override=os.environ.get("COT_STORAGE_ROOT"), drive_module=drive,
    mountpoint=os.environ.get("COT_DRIVE_MOUNTPOINT", "/content/drive"))
STUDY = sessao.Session(REPO, STATE, CODE_COMMIT, CODE_HASHES, on_colab=ON_COLAB, technical_test=TECHNICAL_TEST)
Blocked = sessao.StageBlocked
print("Código:", CODE_COMMIT, "| Estudo:", STUDY_ID, "| Teste técnico:", TECHNICAL_TEST)


In [ ]:
status = STUDY.execute("status")
gate_ok = status["prepared"] and (status["gate"]["passed"] or TECHNICAL_TEST)
registered = status["registration"]["passed"] or TECHNICAL_TEST
if not status["prepared"]:
    print("ETAPA BLOQUEADA: execute primeiro o notebook 00 com o mesmo STUDY_ID.")
elif not gate_ok:
    print("ETAPA BLOQUEADA: triagem DEV", status["gate"]["status"], status["gate"].get("reasons"))
elif not registered:
    print("ETAPA BLOQUEADA: declare o registro público no notebook 00 antes de treinar.")
else:
    try:
        progress = STUDY.execute("train", mode="grpo", max_jobs=MAX_JOBS_PER_SESSION)
        print(f"GRPO: {progress['completed']}/{progress['total']} treinos concluídos ({progress['status']}).")
    except Blocked as reason:
        print("ETAPA BLOQUEADA:", reason)
STUDY.export(download=False)
STUDY.status()


**Situação das etapas.** CONCLUÍDA: arquivos verificados por hash. BLOQUEADA: um critério do
protocolo ou um recurso impede a etapa (o motivo é impresso). NÃO EXECUTADA / PENDENTE: ainda
não iniciada. Reexecutar retoma o mesmo `STUDY_ID`; trocar o ID inicia outro estudo.
Cópias antigas salvas no Colab mantêm o commit antigo: abra sempre pelo link do README.
